# EBM + Random Forest + Logistic/Ridge Baselines on Real Datasets

This notebook demonstrates the evaluation of three baseline methods on the Grinsztajn benchmark datasets:
- **EBM** (Explainable Boosting Machine) — interpretable glass-box model
- **Random Forest** — strong ensemble baseline
- **Logistic/Ridge Regression** — simple linear baseline

The experiment uses pre-assigned K-fold CV splits and records balanced accuracy, AUC, R², fit time, and EBM-specific interpretability metrics (n_terms, n_interaction_terms, n_main_effects).

We demonstrate on the **credit** dataset (binary classification) with a curated 80-example subset.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# interpret — NOT pre-installed on Colab, always install
_pip('interpret==0.6.10')

# Core packages — pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
import gc
import json
import os
import sys
import time

import numpy as np
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, r2_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

## Data Loading

Load the mini demo dataset from GitHub (with local fallback).

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-7dffcf-balance-guided-oblique-trees-signed-spec/main/experiment_iter4_ebm_random_fore/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded {len(data['datasets'])} dataset(s)")
for ds in data["datasets"]:
    print(f"  {ds['dataset']}: {len(ds['examples'])} examples")

## Configuration

All tunable parameters are defined here. Start with minimum values for fast demo execution.

In [ ]:
# ---- Tunable parameters ----
RANDOM_STATE = 42
METHOD_NAMES = ["ebm", "random_forest", "linear"]
N_FOLDS = 5  # number of CV folds to evaluate (max 5, from pre-assigned splits)

# EBM hyperparameters
# Original values: outer_bags=8, interactions=10, max_rounds=5000
EBM_OUTER_BAGS = 2
EBM_INTERACTIONS = 2
EBM_MAX_ROUNDS = 50

# Random Forest hyperparameters
# Original value: n_estimators=100
RF_N_ESTIMATORS = 10

# Linear model hyperparameters
# Original values: max_iter=1000, C=1.0, alpha=1.0
LINEAR_MAX_ITER = 100
LINEAR_C = 1.0
RIDGE_ALPHA = 1.0

## Data Parsing

Convert raw JSON examples into numpy arrays (X, y, folds) suitable for scikit-learn models.

In [ ]:
def parse_dataset(name, examples):
    """Convert list of examples to X (numpy), y (numpy), folds (numpy), metadata."""
    first = examples[0]

    # Extract full feature names from input JSON
    first_input = json.loads(first["input"])
    feature_names_full = list(first_input.keys())

    task_type = first["metadata_task_type"]
    n_classes = first.get("metadata_n_classes", None)
    is_regression = ("regression" in task_type)

    n = len(examples)
    d = len(feature_names_full)
    X = np.zeros((n, d), dtype=np.float64)
    y_raw = []
    folds = np.zeros(n, dtype=np.int32)

    for i, ex in enumerate(examples):
        feats = json.loads(ex["input"])
        for j, fname in enumerate(feature_names_full):
            X[i, j] = float(feats[fname])
        y_raw.append(ex["output"])
        folds[i] = int(ex["metadata_fold"])

    if is_regression:
        y = np.array([float(v) for v in y_raw], dtype=np.float64)
    else:
        unique_labels = sorted(set(y_raw))
        label_map = {lbl: idx for idx, lbl in enumerate(unique_labels)}
        y = np.array([label_map[v] for v in y_raw], dtype=np.int64)
        n_classes = len(unique_labels)

    return {
        "name": name,
        "X": X, "y": y, "folds": folds,
        "feature_names": feature_names_full,
        "task_type": task_type,
        "is_regression": is_regression,
        "n_classes": n_classes,
        "n_samples": n, "n_features": d,
    }

# Parse all datasets from loaded data
datasets = []
for ds_block in data["datasets"]:
    ds = parse_dataset(ds_block["dataset"], ds_block["examples"])
    datasets.append(ds)
    print(f"Parsed {ds['name']}: {ds['n_samples']} x {ds['n_features']}, "
          f"task={ds['task_type']}, n_classes={ds['n_classes']}")

## Model Definitions

Factory function to instantiate EBM, Random Forest, or Logistic/Ridge models using the config parameters.

In [ ]:
def make_model(method_name, is_regression, n_classes=None):
    """Instantiate a fresh model for the given method and task type."""
    if method_name == "ebm":
        if is_regression:
            from interpret.glassbox import ExplainableBoostingRegressor
            return ExplainableBoostingRegressor(
                outer_bags=EBM_OUTER_BAGS,
                interactions=EBM_INTERACTIONS,
                max_rounds=EBM_MAX_ROUNDS,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )
        else:
            from interpret.glassbox import ExplainableBoostingClassifier
            return ExplainableBoostingClassifier(
                outer_bags=EBM_OUTER_BAGS,
                interactions=EBM_INTERACTIONS,
                max_rounds=EBM_MAX_ROUNDS,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )
    elif method_name == "random_forest":
        if is_regression:
            from sklearn.ensemble import RandomForestRegressor
            return RandomForestRegressor(
                n_estimators=RF_N_ESTIMATORS, random_state=RANDOM_STATE, n_jobs=-1)
        else:
            from sklearn.ensemble import RandomForestClassifier
            return RandomForestClassifier(
                n_estimators=RF_N_ESTIMATORS, random_state=RANDOM_STATE, n_jobs=-1)
    elif method_name == "linear":
        if is_regression:
            from sklearn.linear_model import Ridge
            return Ridge(alpha=RIDGE_ALPHA)
        else:
            from sklearn.linear_model import LogisticRegression
            return LogisticRegression(
                max_iter=LINEAR_MAX_ITER, C=LINEAR_C, random_state=RANDOM_STATE)
    else:
        raise ValueError(f"Unknown method: {method_name}")

## Metrics Computation

Compute balanced accuracy, AUC, R², and EBM-specific interpretability metrics for each fold.

In [ ]:
def compute_metrics(model, method_name, X_test, y_test, is_regression, n_classes):
    """Compute all relevant metrics for one fold."""
    metrics = {}

    if is_regression:
        y_pred = model.predict(X_test)
        metrics["r2"] = float(r2_score(y_test, y_pred))
        metrics["balanced_accuracy"] = None
        metrics["auc"] = None
    else:
        y_pred = model.predict(X_test)
        metrics["balanced_accuracy"] = float(
            balanced_accuracy_score(y_test, y_pred))
        metrics["r2"] = None

        # AUC computation
        try:
            if hasattr(model, "predict_proba"):
                y_prob = model.predict_proba(X_test)
            elif hasattr(model, "decision_function"):
                y_prob = model.decision_function(X_test)
            else:
                y_prob = None

            if y_prob is not None:
                if n_classes == 2:
                    if y_prob.ndim == 2:
                        metrics["auc"] = float(
                            roc_auc_score(y_test, y_prob[:, 1]))
                    else:
                        metrics["auc"] = float(
                            roc_auc_score(y_test, y_prob))
                else:
                    metrics["auc"] = float(roc_auc_score(
                        y_test, y_prob,
                        multi_class="ovr", average="weighted"))
            else:
                metrics["auc"] = None
        except Exception as e:
            print(f"    AUC computation failed: {e}")
            metrics["auc"] = None

    # EBM-specific interpretability metrics
    if method_name == "ebm":
        try:
            metrics["n_terms"] = len(model.term_names_)
            metrics["n_interaction_terms"] = sum(
                1 for t in model.term_features_ if len(t) >= 2)
            metrics["n_main_effects"] = sum(
                1 for t in model.term_features_ if len(t) == 1)
        except AttributeError as e:
            print(f"    EBM interpretability metrics failed: {e}")

    return metrics

## Experiment Loop

Train each method (EBM, Random Forest, Linear) on each fold, compute metrics, and aggregate results across folds.

In [ ]:
total_start = time.time()
all_results = {}  # dataset_name -> {method -> {fold_results, aggregate}}

for ds in datasets:
    ds_name = ds["name"]
    X, y, folds_arr = ds["X"], ds["y"], ds["folds"]
    is_reg = ds["is_regression"]
    n_cls = ds["n_classes"]
    unique_folds = sorted(np.unique(folds_arr))
    n_folds = min(N_FOLDS, len(unique_folds))

    print(f"\n{'=' * 60}")
    print(f"Dataset: {ds_name} ({ds['n_samples']} x {ds['n_features']}, "
          f"{'regression' if is_reg else f'{n_cls}-class classification'})")

    ds_results = {}

    for method_name in METHOD_NAMES:
        print(f"\n  Method: {method_name}")
        fold_metrics = []
        method_total_time = 0.0

        for fold_id in unique_folds[:n_folds]:
            test_mask = (folds_arr == fold_id)
            train_mask = ~test_mask
            X_train, X_test = X[train_mask], X[test_mask]
            y_train, y_test = y[train_mask], y[test_mask]

            # Scale for linear models
            if method_name == "linear":
                scaler = StandardScaler()
                X_train_use = scaler.fit_transform(X_train)
                X_test_use = scaler.transform(X_test)
            else:
                X_train_use = X_train
                X_test_use = X_test

            # Train
            t0 = time.time()
            model = make_model(method_name, is_reg, n_cls)
            try:
                model.fit(X_train_use, y_train)
                fit_time = time.time() - t0

                # Metrics
                metrics = compute_metrics(
                    model, method_name, X_test_use, y_test, is_reg, n_cls)
                metrics["fit_time"] = round(fit_time, 3)
                metrics["fold"] = fold_id
                metrics["status"] = "success"

            except Exception as e:
                print(f"    Fold {fold_id} FAILED: {e}")
                fit_time = time.time() - t0
                metrics = {
                    "fold": fold_id, "status": "error",
                    "error": str(e),
                    "fit_time": round(fit_time, 3),
                    "balanced_accuracy": None,
                    "auc": None, "r2": None,
                }

            fold_metrics.append(metrics)
            method_total_time += metrics.get("fit_time", 0)
            print(f"    Fold {fold_id}: "
                  f"bal_acc={metrics.get('balanced_accuracy')}, "
                  f"auc={metrics.get('auc')}, "
                  f"r2={metrics.get('r2')}, "
                  f"time={metrics.get('fit_time', 0):.2f}s")

            del model
            gc.collect()

        # Aggregate across folds
        successful = [m for m in fold_metrics if m["status"] == "success"]
        aggregate = {}
        for metric_key in ["balanced_accuracy", "auc", "r2", "fit_time"]:
            vals = [m[metric_key] for m in successful
                    if m.get(metric_key) is not None]
            if vals:
                aggregate[f"{metric_key}_mean"] = round(float(np.mean(vals)), 6)
                aggregate[f"{metric_key}_std"] = round(float(np.std(vals)), 6)
            else:
                aggregate[f"{metric_key}_mean"] = None
                aggregate[f"{metric_key}_std"] = None
        aggregate["total_fit_time"] = round(method_total_time, 2)
        aggregate["n_successful_folds"] = len(successful)

        # EBM-specific aggregate
        if method_name == "ebm" and successful:
            for ekey in ["n_terms", "n_interaction_terms", "n_main_effects"]:
                evals = [m[ekey] for m in successful if ekey in m]
                if evals:
                    aggregate[f"{ekey}_mean"] = round(float(np.mean(evals)), 1)

        ds_results[method_name] = {
            "fold_results": fold_metrics,
            "aggregate": aggregate,
        }
        print(f"  {method_name} aggregate: {aggregate}")

    all_results[ds_name] = ds_results

total_time = time.time() - total_start
print(f"\nTotal experiment time: {total_time:.1f}s")

## Results Visualization

Summary table of aggregate metrics across methods, plus bar charts comparing balanced accuracy, AUC, and fit time.

In [ ]:
# --- Results Table ---
print("=" * 70)
print("AGGREGATE RESULTS SUMMARY")
print("=" * 70)
header = f"{'Method':<16} {'Bal.Acc (mean)':<16} {'AUC (mean)':<14} {'Fit Time (s)':<14} {'Folds'}"
print(header)
print("-" * 70)

for ds_name, ds_res in all_results.items():
    print(f"\nDataset: {ds_name}")
    for method_name in METHOD_NAMES:
        if method_name not in ds_res:
            continue
        agg = ds_res[method_name]["aggregate"]
        ba = agg.get("balanced_accuracy_mean")
        ba_str = f"{ba:.4f}" if ba is not None else "N/A"
        auc_val = agg.get("auc_mean")
        auc_str = f"{auc_val:.4f}" if auc_val is not None else "N/A"
        ft = agg.get("total_fit_time", 0)
        n_ok = agg.get("n_successful_folds", 0)
        print(f"  {method_name:<14} {ba_str:<16} {auc_str:<14} {ft:<14.2f} {n_ok}")

        # EBM-specific
        if method_name == "ebm":
            nt = agg.get("n_terms_mean")
            ni = agg.get("n_interaction_terms_mean")
            nm = agg.get("n_main_effects_mean")
            if nt is not None:
                print(f"    EBM terms: {nt:.0f} total, {nm:.0f} main, {ni:.0f} interactions")

# --- Bar Charts ---
for ds_name, ds_res in all_results.items():
    methods = [m for m in METHOD_NAMES if m in ds_res]
    ba_vals = []
    auc_vals = []
    ft_vals = []
    ba_stds = []
    auc_stds = []

    for m in methods:
        agg = ds_res[m]["aggregate"]
        ba_vals.append(agg.get("balanced_accuracy_mean") or 0)
        ba_stds.append(agg.get("balanced_accuracy_std") or 0)
        auc_vals.append(agg.get("auc_mean") or 0)
        auc_stds.append(agg.get("auc_std") or 0)
        ft_vals.append(agg.get("total_fit_time") or 0)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    fig.suptitle(f"Results on {ds_name}", fontsize=13, fontweight="bold")
    x = np.arange(len(methods))
    colors = ["#2196F3", "#4CAF50", "#FF9800"]

    # Balanced Accuracy
    axes[0].bar(x, ba_vals, yerr=ba_stds, color=colors, capsize=5)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(methods, rotation=15)
    axes[0].set_ylabel("Balanced Accuracy")
    axes[0].set_title("Balanced Accuracy")
    axes[0].set_ylim(0, 1)

    # AUC
    axes[1].bar(x, auc_vals, yerr=auc_stds, color=colors, capsize=5)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(methods, rotation=15)
    axes[1].set_ylabel("AUC")
    axes[1].set_title("AUC")
    axes[1].set_ylim(0, 1)

    # Fit Time
    axes[2].bar(x, ft_vals, color=colors)
    axes[2].set_xticks(x)
    axes[2].set_xticklabels(methods, rotation=15)
    axes[2].set_ylabel("Total Fit Time (s)")
    axes[2].set_title("Fit Time")

    plt.tight_layout()
    plt.show()

print(f"\nTotal notebook experiment time: {total_time:.1f}s")